# P9-3 · Kiến trúc và hướng mới

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

## Bài tập 3: Lớp MoE tối giản {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class MoE(nn.Module):
    def __init__(self, d=64, d_ff=256, n_experts=8, k=2):
        super().__init__()
        self.k = k
        self.router = nn.Linear(d, n_experts, bias=False)
        self.experts = nn.ModuleList(nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d)) for _ in range(n_experts))

    def forward(self, x):                                  # x: (N, d), N token
        gate = F.softmax(self.router(x), dim=-1)           # (N, E)
        w, idx = gate.topk(self.k, dim=-1)                 # (N, k)
        w = w / w.sum(-1, keepdim=True)
        out = torch.zeros_like(x)
        for e, expert in enumerate(self.experts):
            mask = idx == e                                # (N, k)
            rows = mask.any(-1).nonzero(as_tuple=True)[0]
            if len(rows):
                weight = (w * mask).sum(-1)[rows, None]
                out[rows] += weight * expert(x[rows])
        load = torch.bincount(idx.flatten(), minlength=len(self.experts)).float() / idx.numel()
        return out, load

y, load = MoE()(torch.randn(1000, 64))
print(y.shape, load.round(decimals=3))

torch.Size([1000, 64]) tensor([0.1120, 0.1250, 0.1100, 0.1430, 0.1230, 0.1380, 0.1240, 0.1240])


## Bài tập 4: Router MoE và cân bằng tải {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch
torch.manual_seed(0)
E, N = 8, 1024
logits_lech = torch.randn(N, E) + torch.tensor([2.0, 0, 0, 0, 0, 0, 0, 0])
logits_deu = torch.randn(N, E)

def phan_tich(logits):
    # TODO: trả về (tỉ lệ token của expert bận nhất trong top-2, loss cân bằng tải)
    return 0.0, 0.0

In [ ]:
def phan_tich(logits):
    p = torch.softmax(logits, dim=1)
    top2 = p.topk(2, dim=1).indices
    tai = torch.bincount(top2.flatten(), minlength=E) / N
    f = torch.bincount(top2[:, 0], minlength=E).float() / N
    loss = E * (f * p.mean(0)).sum()
    return tai.max().item(), loss.item()

for ten, lg in [("lệch", logits_lech), ("đều", logits_deu)]:
    bn, l = phan_tich(lg)
    print(f"router {ten}: expert bận nhất nhận {bn:.0%} token (lý tưởng 25%), loss cân bằng {l:.2f}")

router lệch: expert bận nhất nhận 87% token (lý tưởng 25%), loss cân bằng 2.57
router đều: expert bận nhất nhận 28% token (lý tưởng 25%), loss cân bằng 1.00


## Bài tập 5: Best-of-N với bộ kiểm chứng {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from collections import Counter
rng = np.random.default_rng(0)
P_DUNG, SO_BAI = 0.35, 2000

def sinh(N):
    """Trả về mảng N đáp án: 0 là đúng, 1..10 là các đáp án sai."""
    dung = rng.random(N) < P_DUNG
    return np.where(dung, 0, rng.integers(1, 11, N))

def chon(dap_an, cach):
    # TODO: "ngau_nhien" | "da_so" | "verifier"
    return dap_an[0]

In [ ]:
def chon(dap_an, cach):
    if cach == "ngau_nhien":
        return dap_an[0]
    if cach == "da_so":
        return Counter(dap_an).most_common(1)[0][0]
    cham = rng.random(len(dap_an)) < np.where(dap_an == 0, 0.9, 0.1)
    return dap_an[cham.argmax()] if cham.any() else dap_an[0]

kq = {}
for N in (1, 4, 16):
    for cach in ("ngau_nhien", "da_so", "verifier"):
        kq[(cach, N)] = np.mean([chon(sinh(N), cach) == 0 for _ in range(SO_BAI)])
    print(f"N = {N:2d}: ngẫu nhiên {kq[('ngau_nhien', N)]:.2f} | đa số {kq[('da_so', N)]:.2f} | verifier {kq[('verifier', N)]:.2f}")

N =  1: ngẫu nhiên 0.35 | đa số 0.34 | verifier 0.35
N =  4: ngẫu nhiên 0.35 | đa số 0.48 | verifier 0.71
N = 16: ngẫu nhiên 0.37 | đa số 0.87 | verifier 0.83


## Bài tập 6: Phần thưởng kiểm chứng được cho RL {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import re, statistics as st
cau_hoi = "Một bàn giá 1.500.000đ giảm 20%. Giá sau giảm là bao nhiêu nghìn đồng?"
dung = 1200
outputs = ["1500 × 0,8 = 1200. <answer>1200</answer>",
           "Giảm 20% còn 1200 nghìn.",
           "1500 - 20 = 1480 <answer>1480</answer>",
           "Tính: 1500 × 0,2 = 300, 1500 − 300 = 1200 <answer>1200.0</answer>",
           "<answer>1200</answer><answer>1300</answer>",
           "Giá còn <answer> 1 200 </answer>"]

def phan_thuong(output, dap_an_dung):
    # TODO
    return 0.0

In [ ]:
def phan_thuong(output, dap_an_dung):
    the = re.findall(r"<answer>(.*?)</answer>", output, re.S)
    if len(the) != 1:
        return 0.0
    r = 0.1
    try:
        if float(the[0].replace(" ", "").replace(",", ".")) == dap_an_dung:
            r += 1.0
    except ValueError:
        pass
    return r

r = [phan_thuong(o, dung) for o in outputs]
mu, sd = st.mean(r), st.pstdev(r)
adv = [(x - mu) / sd for x in r]
for o, x, a in zip(outputs, r, adv):
    print(f"r = {x:.1f}, advantage {a:+.2f} | {o[:50]}")

r = 1.1, advantage +1.00 | 1500 × 0,8 = 1200. <answer>1200</answer>
r = 0.0, advantage -1.06 | Giảm 20% còn 1200 nghìn.
r = 0.1, advantage -0.87 | 1500 - 20 = 1480 <answer>1480</answer>
r = 1.1, advantage +1.00 | Tính: 1500 × 0,2 = 300, 1500 − 300 = 1200 <answer>
r = 0.0, advantage -1.06 | <answer>1200</answer><answer>1300</answer>
r = 1.1, advantage +1.00 | Giá còn <answer> 1 200 </answer>
